# Training Hybrid Phase-Symbolic Model on ARC

This notebook demonstrates how to train the `HybridPhaseSymbolicARC` model on the ARC-AGI dataset. The model combines **Sparse Fourier Phase Transformers (SFPT)** for pattern extraction with a **Symbolic Program Synthesizer**.

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from rich.console import Console
from rich.panel import Panel

# Add parent directory to path to import nano_moe
sys.path.append('..')

from nano_moe.config import TrainingConfig
from nano_moe.models.phase_symbolic import HybridPhaseSymbolicARC
from nano_moe.inference import system2_reasoning_arc
from nano_moe.data.arc import ARCDataset, collate_arc

console = Console()

## Configuration & Setup

In [ ]:
cfg = TrainingConfig()
cfg.dataset_type = "arc"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

## Load ARC Dataset
This will automatically download the ARC-AGI repository if not present.

In [ ]:
dataset = ARCDataset(data_dir="../data/arc")
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_arc)

print(f"Loaded {len(dataset)} training tasks.")

## Initialize Model
The model consists of:
1. **SparseFourierEmbedding**: Maps grid patterns to frequency domain.
2. **SFPT Encoder**: Extracts features.
3. **PhaseProgramSynthesizer**: Generates symbolic operators (rotations, flips, etc.).
4. **SFPT Decoder**: Reconstructs the output grid.

In [ ]:
model = HybridPhaseSymbolicARC(
    vocab_size=11, # 0-9 colors + padding
    dim=cfg.feature_dim,
    n_layer=cfg.depth,
    n_heads=cfg.n_heads,
    n_freqs=cfg.n_freqs,
    top_k=cfg.top_k,
    expansion=cfg.expansion,
    dropout=0.1,
    n_ops=cfg.n_ops,
    max_program_len=cfg.max_program_len
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
print(model)

## Training Loop

In [ ]:
epochs = 1

for epoch in range(epochs):
    total_loss = 0
    
    for i, batch in enumerate(loader):
        # Prepare batch (padding to max demos)
        max_demos = max(len(item[0]) for item in batch)
        batch_demos_in = []
        batch_demos_out = []
        batch_test_in = []
        batch_test_out = []
        
        for demos_in, demos_out, test_in, test_out in batch:
            curr_demos = len(demos_in)
            pad_count = max_demos - curr_demos
            d_in = torch.stack(demos_in)
            d_out = torch.stack(demos_out)
            if pad_count > 0:
                pad = torch.zeros(pad_count, 30, 30, dtype=torch.long)
                d_in = torch.cat([d_in, pad], dim=0)
                d_out = torch.cat([d_out, pad], dim=0)
            batch_demos_in.append(d_in)
            batch_demos_out.append(d_out)
            batch_test_in.append(test_in)
            batch_test_out.append(test_out)
        
        demos_in_tensor = torch.stack(batch_demos_in).to(device)
        demos_out_tensor = torch.stack(batch_demos_out).to(device)
        test_in_tensor = torch.stack(batch_test_in).to(device)
        test_out_tensor = torch.stack(batch_test_out).to(device)
        
        # Forward Pass
        logits = model.forward_arc(demos_in_tensor, demos_out_tensor, test_in_tensor)
        loss = F.cross_entropy(logits.view(-1, 11), test_out_tensor.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if i % 10 == 0:
            print(f"Epoch {epoch} | Step {i} | Loss: {loss.item():.4f}")
            
    print(f"Epoch {epoch} Complete | Avg Loss: {total_loss / len(loader):.4f}")

## System-2 Inference Demo
We use **System-2 Reasoning** to generate multiple candidate programs and select the best one based on consistency with the demonstration examples.

In [ ]:
with torch.no_grad():
    # Use the last batch from training for demo
    s2_logits = system2_reasoning_arc(model, demos_in_tensor, demos_out_tensor, test_in_tensor, num_samples=5)
    
    # Visualize prediction (simple print of argmax)
    pred_grid = s2_logits.argmax(dim=-1)
    print("Prediction shape:", pred_grid.shape)
    print("Ground Truth shape:", test_out_tensor.shape)
    
    # Calculate accuracy on this batch
    acc = (pred_grid == test_out_tensor).float().mean()
    print(f"Batch Accuracy: {acc.item():.4f}")